# AG07 · Workflow Patterns

**Track:** Agents (lesson 7 of 15) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** AG02 (agent loop), AG03 (tools), AG06 (agentic RAG, where we saw "use the loop only where it pays") · **You will build:** the five classic workflow patterns, then one content pipeline that uses all of them

---
## 🎯 Why this matters
Not every LLM system should be an agent. In *Building effective agents*, Anthropic draws the line between **workflows** (LLMs and tools orchestrated by *predefined code paths*)
and **agents** (the LLM directs its own process). Workflows are cheaper, faster, easier to test and easier to debug; agents trade that for flexibility. Most production "agent" features
are a handful of reusable **workflow patterns**, and knowing them is how you decide *how little autonomy* a task needs.

**Real scenario.** Ledgerly (a fictional invoicing SaaS) ships every fortnight. Marketing needs, from the engineering changelog: a customer-facing release post,
a security notice where relevant, and a social post, all consistent, all accurate, none leaking internal work. Doing it with one giant prompt gives you
a fluent post that quietly omits the security fix. Doing it as a *pipeline of small, checked steps* gives you something you can trust and test.

## 🧭 Learning objectives
By the end you can:
1. Implement prompt chaining (with programmatic gates), routing, parallelization (sectioning and voting), orchestrator-workers and evaluator-optimizer.
2. Choose a pattern from the *shape of the task* (fixed steps? known inputs? unknown subtasks? measurable quality?), and say when a single call is enough.
3. Combine the patterns in one pipeline and account for calls, tokens, estimated cost and wall-clock time per pattern.
4. Add the guard each pattern needs (gates, confidence fallback, vote handling, subtask caps, iteration caps).
5. Explain why "more LLM calls" is a cost you must justify with a measured gain.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: five patterns, one decision rule

| Pattern | Shape | Use when | Costs you |
|---|---|---|---|
| **Prompt chaining** | A → gate → B → gate → C | task splits into fixed sequential steps; each step is easier than the whole | latency; errors propagate unless you gate |
| **Routing** | classify → specialised handler | distinct input categories deserve distinct prompts, tools or *model sizes* | a misroute is a silent wrong answer |
| **Parallelization** | *sectioning*: independent subtasks at once · *voting*: same task N times | speed, separation of concerns (e.g. guardrail beside the main call), or confidence on a risky judgment | N× tokens |
| **Orchestrator-workers** | planner picks subtasks **at run time** → workers → synthesis | you cannot know the subtasks in advance (varies with input) | an extra planning call; runaway decomposition |
| **Evaluator-optimizer** | generate → evaluate → feedback → regenerate | there is a clear quality bar and iteration measurably helps | up to *k* round-trips; may not converge |

**Decision rule** (the order matters):
1. Try **one well-prompted call** (with retrieval/tools if needed). Measure it. If it meets the bar, stop.
2. If it fails because the task has *separable stages*, chain. If inputs are *heterogeneous*, route. If you need *speed or safety separation*, parallelize.
3. If subtasks depend on the input in ways you cannot enumerate, use orchestrator-workers. If quality is checkable and drafts improve with feedback, add evaluator-optimizer.
4. Only when *the number and order of steps cannot be predicted* does an open-ended agent loop (AG02) earn its cost.

**Parallelization versus orchestrator-workers** are easy to confuse: both fan out. In parallelization **you** decide the subtasks in code; in orchestrator-workers **the model** decides them per input.

```
 chain:  ─▶[A]─gate─▶[B]─gate─▶[C]─▶            route:     ─▶[classify]─┬▶[handler 1]
                                                                        ├▶[handler 2]
 sectioning:  ┌▶[guard]─┐   voting: ┌▶[vote]─┐                          └▶[fallback]
           in ┤         ├▶ out    in┼▶[vote]─┼▶ majority
              └▶[draft]─┘           └▶[vote]─┘
 orchestrator-workers:  ─▶[planner]─▶ {worker × N (N decided by the planner)} ─▶[synthesize]
 evaluator-optimizer:   ─▶[generate]─▶[evaluate]──pass──▶ out
                             ▲            │fail + feedback
                             └────────────┘ (max k rounds)
```

**About the offline mode in this lesson.** Pattern code, gates, parsers, caps, metrics and meters are all real. The *model* is played by one Python policy
(`launch_model`) that dispatches on a role in the system prompt; it is deterministic, keyword-based and obviously simpler than Claude. Use it to understand the control flow;
run LIVE to see real model behaviour (the same code, a real `ask()`), and expect different text but the same structure.

## 2 · Build it: shared plumbing, then five patterns
Three small pieces of infrastructure first, because every pattern needs them: an `ask(role, prompt)` helper (so each call has a *role-specific system prompt*), a **meter**
that records every call by pattern (so we can account for cost), and a fixed changelog to work on. Pricing comes from `agentkit.trace.PRICES`; "fast" and "capable" tiers show why routing to a cheaper model matters.

In [ ]:
import json, re, time, threading
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from collections import defaultdict
from agentkit import get_llm, say, PolicyLLM, is_live
from agentkit.trace import PRICES
from agentkit.llm import DEFAULT_MODEL
import contextlib, io

CHANGELOG = [
    {"id": "F-101", "type": "feature", "title": "Auto-Reconcile",
     "details": "Matches bank transactions to open invoices automatically, with a color-coded match status and a daily review queue. Early testers cut month-end reconciliation time substantially."},
    {"id": "F-102", "type": "feature", "title": "Multi-currency invoices",
     "details": "Customers can now invoice in NZD, AUD, USD or GBP with daily exchange rates; supports currency-specific tax lines."},
    {"id": "B-210", "type": "bugfix", "title": "PDF export rounding",
     "details": "Fixed a 1-cent rounding difference on GST-inclusive PDF totals."},
    {"id": "S-007", "type": "security", "title": "Session token rotation",
     "details": "Session tokens now rotate on every sign-in and older tokens expire within 24 hours. Admins should ask staff to sign in again after the update."},
    {"id": "I-330", "type": "internal", "title": "Migrate job queue",
     "details": "Internal queue migration; no customer impact."},
]
print([ (c["id"], c["type"]) for c in CHANGELOG ])

TIERS = {"fast": next(m for m in PRICES if "haiku" in m), "capable": DEFAULT_MODEL}   # ids come from agentkit (PRICES / DEFAULT_MODEL), never typed here
LAT = 0.12                                                          # simulated network latency per OFFLINE call (seconds)

ROLES = {   # one system prompt per role: small, specific prompts beat one giant prompt
 "outliner":  "ROLE:outliner\nYou turn a changelog item into a 3-bullet outline for a customer release note. Output ONLY bullets starting with '- '.",
 "drafter":   "ROLE:drafter\nWrite a customer-facing release note of 60-100 words from the outline. Plain, specific, no hype.",
 "polisher":  "ROLE:polisher\nRewrite in New Zealand English and add a first line 'Subject: <short subject>'. Keep every fact.",
 "router":    "ROLE:router\nClassify the text into exactly one of the given LABELS. Reply JSON: {\"label\": str, \"confidence\": 0-1}. Use label \"unsure\" if no label fits.",
 "handler":   "ROLE:handler\nYou are a specialist support agent for the given LABEL. Answer helpfully and briefly.",
 "voter":     "ROLE:voter\nAnswer YES or NO: does this security change require customers to take action? One word.",
 "guard":     "ROLE:guard\nCheck the text for unverifiable absolute claims, guarantees, or leaked internal details. Reply 'PASS' or 'FLAG: <reason>'.",
 "planner":   "ROLE:planner\nGroup the customer-facing changelog items into 1-5 sections for a release post. Reply JSON: {\"sections\": [{\"heading\": str, \"item_ids\": [str]}]}.",
 "section_writer": "ROLE:section_writer\nWrite one section of the release post (markdown, '## heading' then one short paragraph per item). Use only facts from the items.",
 "generator": "ROLE:generator\nWrite a social post for the launch. If a previous DRAFT and FEEDBACK are given, fix exactly what the feedback says.",
 "critic":    "ROLE:critic\nGiven a DRAFT and the results of automated CHECKS, write 1-3 short, actionable feedback bullets for the writer.",
}

class Meter:
    """Records every model call: which pattern, role, tier, estimated tokens (chars/4)."""
    def __init__(self): self.rows, self.lock = [], threading.Lock()
    def record(self, pattern, role, tier, tin, tout):
        with self.lock: self.rows.append(dict(pattern=pattern, role=role, tier=tier, tin=tin, tout=tout))
    def reset(self): self.rows.clear()
    def summary(self):
        agg = defaultdict(lambda: dict(calls=0, tin=0, tout=0, cost=0.0))
        for r in self.rows:
            a, (pin, pout) = agg[r["pattern"]], PRICES[TIERS[r["tier"]]]
            a["calls"] += 1; a["tin"] += r["tin"]; a["tout"] += r["tout"]; a["cost"] += (r["tin"] * pin + r["tout"] * pout) / 1e6
        return dict(agg)
    def total_calls(self): return len(self.rows)
    def show(self):
        s = self.summary()
        print(f"{'pattern':<22}{'calls':>6}{'tok in':>8}{'tok out':>9}{'est. $':>10}")
        for p, a in s.items(): print(f"{p:<22}{a['calls']:>6}{a['tin']:>8}{a['tout']:>9}{a['cost']:>10.5f}")
        print(f"{'TOTAL':<22}{sum(a['calls'] for a in s.values()):>6}{sum(a['tin'] for a in s.values()):>8}{sum(a['tout'] for a in s.values()):>9}{sum(a['cost'] for a in s.values()):>10.5f}")

METER = Meter()
est = lambda s: max(1, len(s) // 4)
_LLMS: dict = {}

def ask(role: str, user: str, *, pattern: str, tier: str = "capable", max_tokens: int = 700) -> str:
    """One model call with the role's system prompt. Thread-safe so patterns can fan out."""
    if tier not in _LLMS:
        with contextlib.redirect_stdout(io.StringIO()):
            _LLMS[tier] = get_llm(policy=launch_model, model=TIERS[tier])
    out = _LLMS[tier].chat([{"role": "user", "content": user}], system=ROLES[role], max_tokens=max_tokens).text
    METER.record(pattern, role, tier, est(ROLES[role] + user), est(out))
    return out

def jparse(text: str) -> dict:
    """Extract the first JSON object from model text (models sometimes wrap JSON in prose or code fences)."""
    m = re.search(r"\{.*\}", text, re.S)
    if not m: raise ValueError(f"no JSON object in model output: {text[:80]!r}")
    return json.loads(m.group(0))

def pmap(fn, items, workers: int = 8):
    """Run fn over items concurrently, preserving order (threads are fine: model calls are I/O-bound)."""
    with ThreadPoolExecutor(max_workers=workers) as ex: return list(ex.map(fn, items))

### The offline "model"
One policy plays every role. Read it once so you know what each role *would* do, then ignore it. It is intentionally boring: keyword routing, templated writing, and
a `FAULTS` set we use in §4 to inject realistic failures (a chatty model that ignores your output format, a model that never satisfies the critic).

In [ ]:
FAULTS: set[str] = set()
KEYWORDS = {   # lexical stand-in for a classifier model
  "billing": ["charge", "refund", "payment", "card", "billing", "receipt", "invoice"],
  "technical": ["error", "bug", "crash", "export", "500", "login", "slow", "broken", "fails"],
  "sales": ["plan", "pricing", "seats", "quote", "demo", "enterprise", "upgrade", "discount"],
  "feature": ["automatically", "new", "support", "introduc", "matches", "currenc"],
  "bugfix": ["fixed", "bug", "rounding", "error"],
  "security": ["token", "session", "password", "sign-in", "vulnerab", "auth"],
  "internal": ["internal", "migrat", "no customer impact"],
}
US2NZ = {"color": "colour", "organize": "organise", "customize": "customise", "analyze": "analyse", "behavior": "behaviour"}
BANNED = ["revolutionary", "game-changing", "unparalleled", "best-in-class"]

def rubric(text: str) -> dict:
    """Hard, programmatic quality checks for the social post (the evaluator's source of truth)."""
    return {"within 240 chars": len(text) <= 240, "names Auto-Reconcile": "Auto-Reconcile" in text,
            "has a call to action": bool(re.search(r"\bTry\b", text)), "no hype words": not any(b in text.lower() for b in BANNED)}

def launch_model(messages, system, tools):
    time.sleep(LAT)                                        # simulate network latency so parallel speed-ups are measurable
    role = system.split("\n")[0].split(":")[1]
    u = messages[-1]["content"]
    if role == "router":
        labels = re.search(r"LABELS: ([^\n]+)", u).group(1).split(", ")
        body = u.split("TEXT:", 1)[1].lower()
        sc = {l: sum(k in body for k in KEYWORDS[l]) for l in labels}
        total = sum(sc.values())
        if total == 0: return say(json.dumps({"label": "unsure", "confidence": 0.0}))
        top = max(sc, key=sc.get)
        return say(json.dumps({"label": top, "confidence": round(sc[top] / total * min(1.0, 0.5 + 0.25 * total), 2)}))
    if role == "outliner":
        if "outliner_chatty" in FAULTS and "failed:" not in u: return say("Sure! Happy to help. Here is a quick outline of what this release note could cover.")
        if "outliner_stubborn" in FAULTS: return say("Great question! Let me think about how to outline this one.")
        title, details = re.match(r"Item: (.+?) - (.+)", u, re.S).groups()
        return say(f"- Headline: {title}\n- What changed: {details.split('.')[0]}.\n- Who benefits: finance teams and admins using Ledgerly")
    if role == "drafter":
        pts = [l[2:] for l in u.splitlines() if l.startswith("- ")]
        return say(f"We have organized things so that {pts[0].split(': ', 1)[1].lower()} is now live in Ledgerly. " + pts[1].split(': ', 1)[1] +
                   " This update is designed for " + pts[2].split(': ', 1)[1] + ". There is nothing to install: it is available to all plans today and you can customize it from Settings.")
    if role == "polisher":
        body = u
        for us, nz in US2NZ.items(): body = re.sub(us, nz, body, flags=re.I)
        return say("Subject: What's new in Ledgerly\n\n" + body)
    if role == "voter":
        who = int(re.search(r"VOTER (\d)", u).group(1)); low = u.lower()
        needs = ("must" in low) if who == 3 else ("should" in low or "must" in low)    # voter 3 is the "lazy" one on purpose
        return say("YES" if needs else "NO")
    if role == "guard":
        bad = re.search(r"guarantee|100%|never fail|roadmap|JIRA-\d+", u, re.I)
        return say(f"FLAG: contains '{bad.group(0)}'" if bad else "PASS")
    if role == "planner":
        items = re.findall(r"^- (\S+) \| (\w+) \| (.+?) \|", u, re.M)
        order = ["feature", "bugfix", "security"]; names = {"feature": "New features", "bugfix": "Fixes", "security": "Security"}
        secs = [{"heading": names[t], "item_ids": [i for i, ty, _ in items if ty == t]} for t in order if any(ty == t for _, ty, _ in items)]
        if "planner_explodes" in FAULTS: secs = [{"heading": f"Section {i}", "item_ids": [it[0]]} for i, it in enumerate(items * 4)]
        return say("Here is the plan.\n```json\n" + json.dumps({"sections": secs}) + "\n```")
    if role == "section_writer":
        heading = re.search(r"HEADING: (.+)", u).group(1); action = "ACTION_REQUIRED: yes" in u
        parts = [f"## {heading}"]
        for m in re.finditer(r"^- (\S+) \| \w+ \| (.+?) \| (.+)$", u, re.M):
            _, title, details = m.groups()
            parts.append(f"**{title}.** {details}")
        if action: parts.append("**Action required:** please ask your team to sign in again after the update.")
        return say("\n\n".join(parts))
    if role == "generator":
        if "DRAFT:" not in u:
            return say("Introducing Auto-Reconcile: the revolutionary, game-changing way to match bank transactions with open invoices automatically, "
                       "saving finance teams hours every single month, complete with colour-coded match status and a daily review queue!!!")
        draft = re.search(r"DRAFT:\n(.*?)\nFEEDBACK:", u, re.S).group(1); fb = u.split("FEEDBACK:", 1)[1].lower(); fixes = 0
        if "hype" in fb and fixes < 2:
            for b in BANNED: draft = re.sub(r"[, ]*" + b + r",?", "", draft, flags=re.I)
            draft = draft.replace("!!!", "."); fixes += 1
        if "240" in fb and fixes < 2:
            draft = "Auto-Reconcile now matches bank transactions to open invoices automatically."; fixes += 1
        if "call to action" in fb and fixes < 2:
            draft = draft.rstrip() + " Try it in Ledgerly today."; fixes += 1
        return say(draft)
    if role == "critic":
        failed = re.findall(r"FAILED: (.+)", u)
        tips = {"within 240 chars": "Too long for the 240-character limit: keep the single most important benefit.",
                "no hype words": "Remove the hype words; state the concrete benefit instead.",
                "has a call to action": "Add a short call to action beginning with 'Try'.",
                "names Auto-Reconcile": "Name the feature 'Auto-Reconcile' exactly."}
        return say("\n".join("- " + tips.get(f.strip(), f"Fix this failed check: {f.strip()}.") for f in failed) or "- Looks good.")
    if role == "handler":
        label = re.search(r"LABEL: (\w+)", system + u).group(1)
        return say(f"[{label} specialist] Thanks for getting in touch. I've looked into this and here is what I'd do next.")
    raise ValueError(f"unknown role {role}")

print("roles:", sorted(ROLES))

### Pattern 1 · Prompt chaining, with gates
Decompose into steps and put a **gate** (cheap programmatic check) between them. The gate is what turns a chain from "hope" into engineering: it catches a bad
intermediate result *before* the next step builds on it. Here, for one item: **outline → draft → polish to NZ English**. Each gate may allow one retry that feeds the failure reason back to the model.

In [ ]:
class GateError(Exception): pass

@dataclass
class Step:
    name: str
    role: str
    prompt: callable            # prompt(prev_output, original_input) -> str
    gate: callable              # gate(output) -> None (ok) or str (reason it failed)
    tier: str = "capable"

def run_chain(steps: list[Step], original, *, pattern="chaining", retries: int = 1, log: list | None = None):
    out = original
    for st in steps:
        prompt, attempt = st.prompt(out, original), 0
        while True:
            out_try = ask(st.role, prompt, pattern=pattern, tier=st.tier)
            problem = st.gate(out_try)
            if log is not None: log.append((st.name, attempt, "ok" if problem is None else problem))
            if problem is None: out = out_try; break
            if attempt >= retries: raise GateError(f"step '{st.name}' failed its gate after {attempt + 1} tries: {problem}")
            attempt += 1; prompt += f"\n\nYour previous output failed: {problem}. Fix that and reply again in the required format."
    return out

def gate_outline(o):
    bullets = [l for l in o.splitlines() if l.startswith("- ")]
    return None if len(bullets) == 3 else f"expected exactly 3 '- ' bullets, got {len(bullets)}"
def gate_draft(d):
    n = len(d.split()); return None if 25 <= n <= 120 else f"draft is {n} words (need 25-120)"
def gate_polish(p):
    left = [us for us in US2NZ if re.search(us, p, re.I)]
    return None if p.startswith("Subject:") and not left else f"needs 'Subject:' line and NZ spelling (still US: {left})"

RELEASE_NOTE_CHAIN = [
    Step("outline", "outliner", lambda prev, orig: f"Item: {orig['title']} - {orig['details']}", gate_outline),
    Step("draft", "drafter", lambda prev, orig: f"OUTLINE:\n{prev}", gate_draft),
    Step("polish", "polisher", lambda prev, orig: prev, gate_polish, tier="fast"),     # easy step -> cheaper model
]
METER.reset(); log = []
note = run_chain(RELEASE_NOTE_CHAIN, CHANGELOG[0], log=log)
print(note, "\n\ngate log:", log)
assert [s for s, _, r in log] == ["outline", "draft", "polish"] and all(r == "ok" for _, _, r in log)
assert note.startswith("Subject:") and "colour" in note and "color" not in note.lower().replace("colour", "")
assert METER.total_calls() == 3, "a chain makes exactly one call per step (when no gate fails)"

### Pattern 2 · Routing
Classify first, then dispatch to a specialised handler. Two payoffs: **quality** (each handler's prompt/tools are tuned to one job) and **cost** (easy categories go to a cheaper model).
Two hazards: a wrong label is a *silent* wrong answer, so route on **confidence**, with a **human/fallback** lane for "unsure". Support tickets are the textbook case.

In [ ]:
TICKETS = [
    ("I was charged twice for my March subscription, please refund one payment.", "billing"),
    ("The PDF export crashes with a 500 error on invoices over 100 lines.", "technical"),
    ("Do you offer a discount on the enterprise plan for 50 seats?", "sales"),
    ("How do I upgrade to the multi-currency plan?", "sales"),
    ("Login fails after I changed my password.", "technical"),
    ("Can I get a receipt for last month's payment?", "billing"),
    ("hello??", "human"),                                                                 # nothing to go on
    ("My card was declined but the invoice export also fails.", "human"),                 # two categories tie: genuinely ambiguous
]
LABELS = ["billing", "technical", "sales"]
HANDLER_TIER = {"billing": "capable", "technical": "capable", "sales": "fast"}
CONF_MIN = 0.7

def route(text: str, labels: list[str], pattern="routing") -> tuple[str, float]:
    r = jparse(ask("router", f"LABELS: {', '.join(labels)}\nTEXT: {text}", pattern=pattern, tier="fast"))
    if r["label"] not in labels or r["confidence"] < CONF_MIN: return "human", r["confidence"]     # fallback lane
    return r["label"], r["confidence"]

def handle_ticket(text: str):
    label, conf = route(text, LABELS)
    if label == "human": return label, conf, "Escalated to a human agent."
    return label, conf, ask("handler", f"LABEL: {label}\nTicket: {text}", pattern="routing", tier=HANDLER_TIER[label])

METER.reset(); results = [handle_ticket(t) + (gold,) for t, gold in TICKETS]
for (text, _), (label, conf, reply, gold) in zip(TICKETS, results):
    print(f"{'OK ' if label == gold else 'XX '} gold={gold:<9} routed={label:<9} conf={conf:.2f}  {text[:55]}")
acc = sum(label == gold for label, _, _, gold in results) / len(results)
print(f"routing accuracy: {acc:.0%}")
assert acc == 1.0, "clear tickets route correctly; the two ambiguous ones must go to the human lane, not to a guess"

# cost of routing: sales tickets used the fast tier. What if every handler call used the capable tier?
s = METER.summary()["routing"]
all_capable = sum((r["tin"] * PRICES[TIERS["capable"]][0] + r["tout"] * PRICES[TIERS["capable"]][1]) / 1e6 for r in METER.rows)
print(f"estimated cost routed: ${s['cost']:.5f} | everything on the capable tier: ${all_capable:.5f}")
assert s["cost"] < all_capable

### Pattern 3 · Parallelization: sectioning and voting
**Sectioning**: independent subtasks run at the same time. A classic use is a **guardrail beside the main call**: one call drafts, another checks, and the
check does not slow the answer down (and a guard that is its own call is not distracted by the writing task).
**Voting**: run the *same* judgment several times and aggregate, when a single call is too noisy on a high-stakes decision. Both multiply tokens; we measure the latency win.

In [ ]:
def sequential_vs_parallel():
    draft_in = f"OUTLINE:\n- Headline: Auto-Reconcile\n- What changed: Matches bank transactions to open invoices.\n- Who benefits: finance teams"
    tasks = [lambda: ask("drafter", draft_in, pattern="sectioning"), lambda: ask("guard", CHANGELOG[0]["details"], pattern="sectioning"),
             lambda: ask("guard", CHANGELOG[1]["details"], pattern="sectioning"), lambda: ask("guard", CHANGELOG[2]["details"], pattern="sectioning")]
    t0 = time.perf_counter(); seq = [t() for t in tasks]; t_seq = time.perf_counter() - t0
    t0 = time.perf_counter(); par = pmap(lambda t: t(), tasks); t_par = time.perf_counter() - t0
    return seq, par, t_seq, t_par

METER.reset()
seq, par, t_seq, t_par = sequential_vs_parallel()
print(f"sectioning: sequential {t_seq:.2f}s vs parallel {t_par:.2f}s (4 calls each)")
assert par == seq and t_par < 0.6 * t_seq, "same results, lower wall-clock"

def aggregate(votes: list[str], tie: str = "no") -> bool:
    """Strict majority of YES. A tie (possible only with an even panel) goes to `tie`: an explicit, written-down decision."""
    yes = sum(v.strip().upper().startswith("YES") for v in votes)
    return True if yes * 2 > len(votes) else False if yes * 2 < len(votes) else tie == "yes"

def vote(item: dict, n: int = 3, tie: str = "no") -> tuple[bool, list[str]]:
    votes = pmap(lambda i: ask("voter", f"VOTER {i}\nTitle: {item['title']}\nDetails: {item['details']}", pattern="voting", tier="fast"), range(1, n + 1))
    return aggregate(votes, tie), votes

METER.reset(); decision, votes = vote(CHANGELOG[3])
print("S-007 votes:", votes, "->", "ACTION REQUIRED" if decision else "no action")
assert votes == ["YES", "YES", "NO"] and decision is True and METER.total_calls() == 3
assert vote(CHANGELOG[2])[0] is False             # the rounding bugfix: all voters say NO

### Pattern 4 · Orchestrator-workers
When you cannot enumerate the subtasks in advance, let a **planner** decide them *per input*, run **workers** on each, and **synthesise**. Contrast with sectioning: the number
of worker calls below is a function of the changelog. The planner's output is *untrusted structure*: parse it, validate it (every id exists and is covered exactly once) and **cap** the section count.

In [ ]:
MAX_SECTIONS = 5

def plan_sections(items: list[dict]) -> list[dict]:
    listing = "\n".join(f"- {i['id']} | {i['type']} | {i['title']} | {i['details']}" for i in items)
    plan = jparse(ask("planner", f"ITEMS:\n{listing}", pattern="orchestrator"))["sections"]
    ids = [i for s in plan for i in s["item_ids"]]
    if len(plan) > MAX_SECTIONS: raise GateError(f"planner produced {len(plan)} sections (cap {MAX_SECTIONS})")
    if sorted(ids) != sorted(i["id"] for i in items): raise GateError(f"plan must cover each item exactly once; got {sorted(ids)}")
    return plan

def write_sections(items: list[dict], plan: list[dict], action_required: set[str] = frozenset()) -> str:
    by_id = {i["id"]: i for i in items}
    def worker(sec):
        lines = "\n".join(f"- {i} | {by_id[i]['type']} | {by_id[i]['title']} | {by_id[i]['details']}" for i in sec["item_ids"])
        flag = "yes" if set(sec["item_ids"]) & action_required else "no"
        return ask("section_writer", f"HEADING: {sec['heading']}\nACTION_REQUIRED: {flag}\nITEMS:\n{lines}", pattern="workers")
    return "\n\n".join(pmap(worker, plan))                # synthesis here is deterministic assembly; it can also be one more model call

customer_facing = [c for c in CHANGELOG if c["type"] != "internal"]
only_fixes = [c for c in CHANGELOG if c["type"] == "bugfix"]
for name, items in [("full changelog", customer_facing), ("a bugfix-only release", only_fixes)]:
    METER.reset(); plan = plan_sections(items); post = write_sections(items, plan)
    print(f"{name}: planner chose {len(plan)} section(s) -> model calls = {METER.total_calls()} (1 planner + {len(plan)} workers)")
    assert METER.total_calls() == 1 + len(plan)
assert len(plan_sections(customer_facing)) == 3 and len(plan_sections(only_fixes)) == 1
print("\n" + write_sections(only_fixes, plan_sections(only_fixes)))

### Pattern 5 · Evaluator-optimizer
One call generates; another evaluates against a **rubric** and returns feedback; the generator revises. It pays off only when (a) you can state the quality bar
and (b) revision demonstrably helps. Two design rules we follow here: **pass/fail is decided by code** (`rubric()`) wherever the criteria are checkable,
with the model writing the *feedback* in words; and there is a **hard iteration cap** with *best-so-far* returned if the bar is never met.

In [ ]:
def optimize(max_iters: int = 4, pattern="eval-opt"):
    draft, history = None, []
    for i in range(1, max_iters + 1):
        prompt = "Write a launch post (max 240 characters) announcing Auto-Reconcile from Ledgerly." + (f"\nDRAFT:\n{draft}\nFEEDBACK:\n{feedback}" if draft else "")
        draft = ask("generator", prompt, pattern=pattern)
        checks = rubric(draft); score = sum(checks.values()) / len(checks)
        history.append((i, score, draft))
        if all(checks.values()): return draft, history, True
        failed = "\n".join(f"FAILED: {k}" for k, ok in checks.items() if not ok)
        feedback = ask("critic", f"DRAFT:\n{draft}\nCHECKS:\n{failed}", pattern=pattern, tier="fast")
    best = max(history, key=lambda h: h[1])               # never converged: return the best attempt, flagged
    return best[2], history, False

METER.reset(); tweet, hist, ok = optimize()
for i, score, d in hist: print(f"round {i}: score {score:.2f} | {len(d):>3} chars | {d[:90]}{'...' if len(d) > 90 else ''}")
assert ok and len(hist) <= 4 and all(rubric(tweet).values())
scores = [s for _, s, _ in hist]; assert scores == sorted(scores) and scores[0] < 1.0 == scores[-1], "quality should improve monotonically here"
print("calls:", METER.total_calls(), "(generator + critic per failed round, generator once more at the end)")

## 3 · Use it: one content pipeline, all five patterns
Now compose them on the real task. Read the pipeline as *a chain of stages with gates* (the chain is the spine; every other pattern is a stage):

| Stage | Pattern | What it does | Guard |
|---|---|---|---|
| 1 | **Routing** | classify each changelog item (feature / bugfix / security / internal); drop internal work | confidence floor; unsure → human review list |
| 2 | **Voting** | for each security item, 3 voters decide "customers must act?" | strict majority |
| 3 | **Orchestrator** | planner groups the customer-facing items into sections | validate coverage; cap sections |
| 4 | **Workers (sectioning)** | one writer per section, in parallel | – |
| 5 | **Chaining** | polish the assembled post to NZ English | gate: Subject line + no US spellings |
| 6 | **Parallel sectioning** | *guard* the post **while** the evaluator-optimizer writes the social post | guard must PASS |
| 7 | **Evaluator-optimizer** | social post that meets the rubric | cap on rounds |

Then a final **coverage gate** in code: every customer-facing item must appear in the post, and no internal item may.

In [ ]:
def launch_pipeline(changelog: list[dict]) -> dict:
    t0, phases = time.perf_counter(), {}
    # 1. routing (per item, in parallel; the classifier sees only the text, never our 'type' field)
    labels = ["feature", "bugfix", "security", "internal"]
    routed = pmap(lambda it: route(f"{it['title']}: {it['details']}", labels, pattern="routing"), changelog)
    items = [dict(it, type=lab) for it, (lab, _) in zip(changelog, routed)]
    needs_review = [it["id"] for it, (lab, _) in zip(changelog, routed) if lab == "human"]
    customer = [it for it in items if it["type"] in ("feature", "bugfix", "security")]
    phases["route"] = time.perf_counter() - t0
    # 2. voting on security items
    action = {it["id"] for it in customer if it["type"] == "security" and vote(it)[0]}
    # 3-4. orchestrator + workers
    plan = plan_sections(customer); body = write_sections(customer, plan, action)
    # 5. chain step with a gate: polish
    log = []
    post = run_chain([Step("polish", "polisher", lambda prev, orig: prev, gate_polish, tier="fast")], body, log=log, pattern="chaining")
    # 6. parallel sectioning: guard || social post (evaluator-optimizer)
    t1 = time.perf_counter()
    guard_out, (tweet, hist, tweet_ok) = pmap(lambda f: f(), [lambda: ask("guard", post, pattern="guard", tier="fast"), optimize])
    phases["guard||tweet"] = time.perf_counter() - t1
    # final coverage gate (code, not a model)
    missing = [it["id"] for it in customer if it["title"].lower() not in post.lower()]
    leaked = [it["id"] for it in items if it["type"] == "internal" and it["title"].lower() in post.lower()]
    return dict(post=post, tweet=tweet, tweet_ok=tweet_ok, guard=guard_out, plan=plan, action=action, needs_review=needs_review,
                missing=missing, leaked=leaked, routed={it["id"]: it["type"] for it in items}, seconds=time.perf_counter() - t0, phases=phases)

METER.reset()
R = launch_pipeline(CHANGELOG)
print(R["post"]); print("\nSOCIAL:", R["tweet"]); print("\nguard:", R["guard"], "| action-required items:", R["action"], "| routed:", R["routed"])
assert R["routed"] == {c["id"]: c["type"] for c in CHANGELOG}, "router must recover the true labels without seeing them"
assert R["missing"] == [] and R["leaked"] == [] and R["guard"] == "PASS" and R["tweet_ok"] and R["needs_review"] == []
assert R["action"] == {"S-007"} and "Action required" in R["post"] and "colour" in R["post"] and "Migrate job queue" not in R["post"]
assert R["post"].startswith("Subject:")

### Account for it
Every pattern added calls. Which ones, and what did they buy? (Tokens are the `chars/4` estimate; dollars use `PRICES`; offline no money is spent, this is what the *same* calls would cost.)
We also compare against the naive alternative of doing the same work with one call per changelog item and nothing else.

In [ ]:
METER.show()
n_items = len(CHANGELOG)
print(f"\nwall-clock {R['seconds']:.2f}s | sequential-equivalent ≈ {METER.total_calls() * LAT:.2f}s of model latency "
      f"({METER.total_calls()} calls x {LAT}s simulated)")
by = METER.summary()
assert by["routing"]["calls"] == n_items                          # one classification per item
assert by["voting"]["calls"] == 3 and by["orchestrator"]["calls"] == 1 and by["workers"]["calls"] == len(R["plan"])
assert by["guard"]["calls"] == 1 and by["eval-opt"]["calls"] >= 3 and by["chaining"]["calls"] == 1
assert METER.total_calls() > n_items, "workflows buy reliability with extra calls; the point is to know exactly how many"
assert R["seconds"] < METER.total_calls() * LAT * 0.8, "parallel stages must beat the sequential-equivalent latency"

**Reading the table.** The evaluator-optimizer loop and the per-item routing calls dominate the *call count*, but routing uses the fast tier, so it barely moves the bill.
The parallel stages cost tokens but save wall-clock time. If a stage's gain is not measurable (does voting ever change the decision? does the critic loop ever fire?), **delete it**.
That audit, "which of my five patterns earns its calls on real data?", is the habit this lesson is trying to build; AG14 turns it into an eval harness.

## 4 · Break it
Each pattern has a characteristic failure. Run them and note the guard.

### 4a · A chain without gates propagates garbage
A chatty model ignores the "bullets only" instruction ("Sure! Happy to help..."). Without the gate, the drafter would happily write a note *from that sentence*. With it, the failure is caught
at the step that caused it; a retry with the reason fed back fixes a one-off, and a persistent failure raises a clear error instead of shipping nonsense.

In [ ]:
FAULTS.add("outliner_chatty")
log = []
try:
    out = run_chain(RELEASE_NOTE_CHAIN, CHANGELOG[2], log=log)
finally:
    FAULTS.discard("outliner_chatty")
print("recovered after retry:", log[:3])
assert log[0] == ("outline", 0, "expected exactly 3 '- ' bullets, got 0") and log[1] == ("outline", 1, "ok")

FAULTS.add("outliner_stubborn")
try:
    run_chain(RELEASE_NOTE_CHAIN, CHANGELOG[2]); raise AssertionError("should have raised")
except GateError as e:
    print("persistent failure ->", e)
finally:
    FAULTS.discard("outliner_stubborn")

### 4b · Router: the silent misroute
Look at the two ambiguous tickets in §2: a classifier that *always* picks a label would send "card declined and export fails" to billing or technical with false confidence.
Our guard is a **confidence floor** plus a human lane. Sweep the floor to see the trade-off: too low and ambiguity leaks through; too high and you escalate everything.

In [ ]:
for floor in (0.0, 0.7, 0.95):
    CONF_MIN = floor
    outcome = [route(t, LABELS)[0] for t, _ in TICKETS]
    wrong = sum(o != g and o != "human" for o, (_, g) in zip(outcome, TICKETS)); escal = sum(o == "human" for o in outcome)
    print(f"floor {floor:.2f}: {escal} escalated to human, {wrong} wrongly auto-routed")
CONF_MIN = 0.7
assert sum(route(t, LABELS)[0] == "human" for t, _ in TICKETS) == 2

### 4c · Evaluator-optimizer that never converges, and runaway orchestration
*Loop that cannot satisfy its critic:* the cap returns the **best attempt, flagged**, rather than looping or silently shipping the last one.
*Orchestrator that over-decomposes:* the planner's output is untrusted input: cap and validate it, because each section is a paid model call.

In [ ]:
orig_rubric = rubric
def impossible(text): return {**orig_rubric(text), "contains a unicorn": "unicorn" in text}         # a bar no draft can meet
rubric = impossible
try:
    METER.reset(); tweet, hist, ok = optimize(max_iters=3)
finally:
    rubric = orig_rubric
print(f"converged={ok} after {len(hist)} rounds; returned best-so-far (score {max(s for _, s, _ in hist):.2f}); calls={METER.total_calls()}")
assert ok is False and len(hist) == 3 and METER.total_calls() <= 3 * 2

FAULTS.add("planner_explodes")
try:
    plan_sections(customer_facing); raise AssertionError("cap should have fired")
except GateError as e:
    print("orchestrator guard ->", e)
finally:
    FAULTS.discard("planner_explodes")

### 4d · Voting with an even number of voters
With 2 voters a 1-1 split has no majority. Whatever you do (default to "yes, notify customers"? to "no"?) must be a *decision you made*, because it is an input to a safety notice.
`aggregate()` takes the tie rule as an argument; for a *security* notice you would likely default to the safe side (`tie="yes"`: notify), whereas for a marketing claim you might default to `"no"`:

In [ ]:
split = ["YES", "NO"]
print("1-1 tie, tie='no' :", aggregate(split, tie="no"), "| tie='yes':", aggregate(split, tie="yes"))
assert aggregate(split, "no") is False and aggregate(split, "yes") is True
assert aggregate(["YES", "YES", "NO"]) is True and aggregate(["NO", "NO", "YES"]) is False   # odd panels never tie

> **Principle:** *every pattern is a loop or a fan-out, so every pattern needs a bound and a fallback.* Chains need gates, routers need a floor and a human lane,
> votes need a tie rule, planners need caps, optimizers need an iteration limit and a best-so-far.

## 5 · Exercises
**Practice (30 min)**
1. Add a fifth changelog item (type `"pricing"`) and extend `KEYWORDS`, the planner's section order and the writer. Re-run the pipeline; which asserts must change and why?
2. In `run_chain`, log the elapsed time per step. Which step would you move to the `fast` tier next, and what check would tell you the quality did not drop?

**Stretch (60 min)**
3. Add a **second guard** (PII/legal) in parallel with the existing one and show wall-clock time is unchanged while calls increase by one.
4. Implement **best-of-n** (parallel voting on *quality*): generate 3 social posts at once, score each with `rubric`, and return the best. Compare calls and score distribution against the evaluator-optimizer loop. When does each win?

**Challenge (2 h, needs a key)**
5. Run `launch_pipeline` LIVE. Record per-pattern calls, tokens and cost, and the failure rate of each gate over 10 different changelogs you write (include a tricky security item and an item that is ambiguous between bugfix and security). Which pattern, if removed, changes the outcome least? Remove it and defend the decision with numbers.

In [ ]:
# Starter for exercise 4 - fill in and make the assertions pass
def best_of_n(n: int = 3) -> tuple[str, float]:
    """Generate n posts in parallel, return the best by rubric score."""
    # TODO: pmap over n ask("generator", ..., pattern="best-of-n") calls; score with rubric(); return (best_text, best_score)
    raise NotImplementedError

# METER.reset(); text, score = best_of_n(3)
# assert METER.total_calls() == 3 and 0 <= score <= 1
# print("with a deterministic generator all 3 drafts are identical: why does best-of-n need *sampling diversity* (temperature) to help?")

## ✅ Checkpoint
1. What is the difference between *sectioning* and *orchestrator-workers*, given that both fan out to several model calls?
2. Name the guard that each of the five patterns needs, in one phrase each.
3. When is it *wrong* to add an evaluator-optimizer loop?
4. Your pipeline's p50 latency doubled after adding a chain step that "improves tone". How do you decide whether to keep it?

<details><summary>Answers</summary>

1. Sectioning: the programmer fixes the subtasks in code (known at design time). Orchestrator-workers: a planner model chooses the subtasks per input at run time, so the number of worker calls varies and the plan itself must be validated and capped.
2. Chaining: programmatic gates between steps. Routing: confidence floor + fallback/human lane. Parallelization: tie/aggregation rule (voting) and independence of subtasks (sectioning). Orchestrator-workers: validate and cap the plan. Evaluator-optimizer: iteration cap + best-so-far.
3. When there is no checkable quality bar (the evaluator is then just a second opinion with the same blind spots), when one revision never helps measurably, or when latency/cost budgets cannot absorb the extra round trips.
4. A/B it on a held-out set with the metric that matters (e.g. rubric pass rate or human preference) and compare to the added latency and cost. If the metric gain is not measurable or is smaller than the cost, remove the step; consider moving it to the cheaper tier or running it in parallel instead.
</details>

## 📌 Key takeaways
- Start with **one call**; add structure only where a measured failure demands it. Workflows beat agents on cost, speed and testability whenever the steps are knowable.
- **Chaining** = steps + gates. **Routing** = classify + specialise (+ cheaper models) + fallback. **Parallelization** = sectioning for speed/separation, voting for confidence.
- **Orchestrator-workers** when subtasks depend on the input; **evaluator-optimizer** when quality is checkable and revision helps. Prefer *code* for pass/fail, models for feedback.
- Every pattern needs a **bound and a fallback**. Meter calls per pattern and **delete the ones that do not pay for themselves**.

## 📚 Further reading
- Anthropic Engineering, *Building effective agents* (2024), anthropic.com/engineering/building-effective-agents: the source of this five-pattern taxonomy
- Wang et al., *Self-Consistency Improves Chain of Thought Reasoning in Language Models* (2022), arXiv:2203.11171 (the voting idea applied to reasoning)
- Madaan et al., *Self-Refine: Iterative Refinement with Self-Feedback* (2023), arXiv:2303.17651 (generate, critique, revise)
- Wei et al., *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (2022), arXiv:2201.11903 (decomposing a task *inside* one prompt, the cheapest form of a chain)

**Next → AG08 · LangGraph** (the same patterns as explicit graphs with state, checkpoints, resume, and a mini graph engine you build first)